# 11 — Knowledge Distillation: Train ONNX Student Models

Train lightweight signal models using GPT-4o teacher pseudo-labels from notebook 10.

## Models Trained
| Model | Backbone | Signals | Handcrafted Features |
|-------|----------|---------|---------------------|
| Structure | MobileNetV3-Large | structure (+ pore, texture aux) | Green channel CLAHE |
| Hydration | EfficientNet-B0 | hydration | Gabor (24d) + LBP (18d) + specular (2d) = 44d |
| Elasticity | EfficientNet-B0 | elasticity | Frangi (9d) + landmark geometry (5d) = 14d |
| Multi-Signal | EfficientNet-B0 | all 5 signals | Gabor + LBP + Frangi + specular = 58d |

## Distillation Strategy
- **Loss**: Smooth L1 (Huber) — robust to teacher label noise
- **Label smoothing**: Add N(0, 2) noise during training to prevent overfitting to teacher artifacts
- **Augmentations**: Skin-specific (color jitter, blur, noise, cutout, lighting)
- **Schedule**: Cosine annealing with warmup
- **Output**: ONNX models matching backend inference expectations

In [ ]:
# Install dependencies (Colab — use GPU runtime)
!pip install -q torch torchvision timm "albumentations>=1.3.0" onnx onnxruntime opencv-python scikit-image scikit-learn

In [ ]:
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.feature import local_binary_pattern
from skimage.filters import frangi
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

# ── Config ──────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Glowlytics')
except Exception:
    BASE_DIR = Path('./data')

LABELS_DIR = BASE_DIR / 'datasets' / 'teacher_labels'
MODELS_DIR = BASE_DIR / 'models' / 'distilled'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 40
LR = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
LABEL_NOISE_STD = 2.0  # Gaussian noise on teacher labels (regularization)
NUM_WORKERS = 4

SIGNALS = ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']

## 1. Handcrafted Feature Extraction

These match the backend (`image-processing.js`) so student models learn the same
feature space used at inference time.

In [ ]:
# ── Gabor filter bank (matches backend) ────────────────────────────
def build_gabor_bank(orientations: int = 4, frequencies: list = None) -> list:
    if frequencies is None:
        frequencies = [0.1, 0.25, 0.4]
    kernels = []
    for theta_idx in range(orientations):
        theta = theta_idx * np.pi / orientations
        for freq in frequencies:
            kernel = cv2.getGaborKernel(
                ksize=(31, 31), sigma=4.0, theta=theta,
                lambd=1.0 / freq, gamma=0.5, psi=0
            )
            kernels.append(kernel)
    return kernels

GABOR_BANK = build_gabor_bank()


def extract_gabor_features(gray: np.ndarray) -> np.ndarray:
    """24-dim: 12 filters x (mean, std)."""
    features = []
    for kernel in GABOR_BANK:
        response = cv2.filter2D(gray, cv2.CV_64F, kernel)
        features.extend([response.mean(), response.std()])
    return np.array(features, dtype=np.float32)


def extract_lbp_features(gray: np.ndarray, radius: int = 2, n_points: int = 16) -> np.ndarray:
    """18-dim: uniform LBP histogram."""
    lbp = local_binary_pattern(gray, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=n_points + 2, range=(0, n_points + 2), density=True)
    return hist.astype(np.float32)


def extract_specular_features(image: np.ndarray, threshold: int = 220) -> np.ndarray:
    """2-dim: highlight ratio, spatial spread."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
    highlights = (gray > threshold).astype(np.float32)
    ratio = highlights.mean()
    if highlights.sum() > 0:
        coords = np.argwhere(highlights > 0)
        spread = coords.std(axis=0).mean() / max(gray.shape)
    else:
        spread = 0.0
    return np.array([ratio, spread], dtype=np.float32)


def extract_frangi_features(gray: np.ndarray) -> np.ndarray:
    """9-dim: 3 ROI x (density, intensity, max_response).
    Without landmarks, we use fixed forehead/cheek crops."""
    h, w = gray.shape
    rois = [
        gray[0:h//4, w//4:3*w//4],           # Forehead
        gray[h//4:h//2, 0:w//3],              # Left cheek area
        gray[h//4:h//2, 2*w//3:w],            # Right cheek area
    ]
    features = []
    for roi in rois:
        if roi.size < 100:
            features.extend([0.0, 0.0, 0.0])
            continue
        resp = frangi(
            roi.astype(np.float64) / 255.0,
            sigmas=range(1, 5), alpha=0.5, beta=0.5, gamma=15,
            black_ridges=True,
        )
        density = float((resp > 0.01).mean())
        intensity = float(resp[resp > 0.01].mean()) if density > 0 else 0.0
        features.extend([density, intensity, float(resp.max())])
    return np.array(features, dtype=np.float32)


def extract_landmark_geometry_proxy(gray: np.ndarray) -> np.ndarray:
    """5-dim: proxy geometry ratios from image proportions.
    Without a face detector, use fixed-crop intensity ratios as proxy."""
    h, w = gray.shape
    forehead = gray[0:h//5, w//4:3*w//4].mean() / 255.0
    left_eye = gray[h//5:2*h//5, w//8:3*w//8].mean() / 255.0
    right_eye = gray[h//5:2*h//5, 5*w//8:7*w//8].mean() / 255.0
    nose = gray[2*h//5:3*h//5, 3*w//8:5*w//8].mean() / 255.0
    chin = gray[4*h//5:h, w//4:3*w//4].mean() / 255.0
    return np.array([forehead, left_eye, right_eye, nose, chin], dtype=np.float32)


# Feature dimensions
HYDRATION_DIM = 24 + 18 + 2   # Gabor + LBP + specular = 44
ELASTICITY_DIM = 9 + 5         # Frangi + landmark proxy = 14
MULTI_DIM = 24 + 18 + 2 + 9 + 5  # All combined = 58

print(f'Feature dims: hydration={HYDRATION_DIM}, elasticity={ELASTICITY_DIM}, multi={MULTI_DIM}')

## 2. Dataset

In [ ]:
class TeacherLabeledDataset(Dataset):
    """Dataset loading teacher-labeled images for student training.

    Extracts handcrafted features on-the-fly and returns image tensor + features + labels.
    """

    def __init__(self, split_path: str, signals: list[str], transform=None,
                 label_noise_std: float = 0.0, feature_mode: str = 'all'):
        with open(split_path) as f:
            self.labels = json.load(f)
        self.signals = signals
        self.transform = transform
        self.label_noise_std = label_noise_std
        self.feature_mode = feature_mode  # 'hydration', 'elasticity', 'all', 'none'

        # Validate paths exist
        valid = [l for l in self.labels if os.path.exists(l['image_path'])]
        if len(valid) < len(self.labels):
            print(f'  Warning: {len(self.labels) - len(valid)} images not found, using {len(valid)}')
        self.labels = valid

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        entry = self.labels[idx]
        image = cv2.imread(entry['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))

        # Apply augmentations FIRST so both CNN and handcrafted features
        # see the same image (matches inference where neither is augmented)
        if self.transform:
            augmented = self.transform(image=image)
            image_tensor = augmented['image']
            # Reconstruct the augmented image for feature extraction:
            # undo normalization to get pixel values back in [0, 255]
            aug_np = image_tensor.numpy().transpose(1, 2, 0)  # CHW -> HWC
            aug_np = (aug_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])) * 255
            aug_np = np.clip(aug_np, 0, 255).astype(np.uint8)
        else:
            image_tensor = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            aug_np = image

        gray = cv2.cvtColor(aug_np, cv2.COLOR_RGB2GRAY)

        # Extract handcrafted features from the same (possibly augmented) image
        if self.feature_mode == 'hydration':
            features = np.concatenate([
                extract_gabor_features(gray),
                extract_lbp_features(gray),
                extract_specular_features(aug_np),
            ])
        elif self.feature_mode == 'elasticity':
            features = np.concatenate([
                extract_frangi_features(gray),
                extract_landmark_geometry_proxy(gray),
            ])
        elif self.feature_mode == 'all':
            features = np.concatenate([
                extract_gabor_features(gray),
                extract_lbp_features(gray),
                extract_specular_features(aug_np),
                extract_frangi_features(gray),
                extract_landmark_geometry_proxy(gray),
            ])
        else:
            features = np.zeros(1, dtype=np.float32)  # placeholder

        # Extract signal labels
        label_vals = [entry['signals'][sig] for sig in self.signals]
        labels = torch.tensor(label_vals, dtype=torch.float32)

        # Add label noise during training (regularization against teacher artifacts)
        if self.label_noise_std > 0:
            noise = torch.randn_like(labels) * self.label_noise_std
            labels = torch.clamp(labels + noise, 0, 100)

        return image_tensor, torch.from_numpy(features), labels


# ── Augmentation pipelines ─────────────────────────────────────────
# Skin-specific: color jitter for different lighting, blur for camera quality,
# noise for sensor variation, cutout for occlusion robustness
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
    ], p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.2),
    A.GaussNoise(p=0.2),
    A.CoarseDropout(p=0.15),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

print('Dataset and augmentation pipeline ready.')

## 3. Model Definitions

Four student architectures matching the backend ONNX inference contracts.

In [ ]:
class StructureStudent(nn.Module):
    """Structure signal: MobileNetV3-Large → 3 outputs (pore, texture, structure).

    Pore and texture are auxiliary tasks that regularize training.
    Only structure_score is used at inference — the aux heads are dropped in ONNX export.
    Input: image only (green channel CLAHE preprocessing done in backend).
    """

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features

        self.shared = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
        )
        # Main head
        self.structure_head = nn.Linear(256, 1)
        # Aux heads (dropped at export)
        self.pore_head = nn.Linear(256, 1)
        self.texture_head = nn.Linear(256, 1)

    def forward(self, x):
        feat = self.backbone(x)
        shared = self.shared(feat)
        structure = self.structure_head(shared)
        pore = self.pore_head(shared)
        texture = self.texture_head(shared)
        return torch.cat([pore, texture, structure], dim=-1)  # [B, 3]


class HydrationStudent(nn.Module):
    """Hydration signal: EfficientNet-B0 + 44d handcrafted → hydration score."""

    def __init__(self, handcrafted_dim: int = 44, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        cnn_dim = self.backbone.num_features  # 1280

        self.hc_fc = nn.Sequential(
            nn.Linear(handcrafted_dim, 64),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.head = nn.Sequential(
            nn.Linear(cnn_dim + 64, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)
        hc_feat = self.hc_fc(handcrafted)
        return self.head(torch.cat([cnn_feat, hc_feat], dim=-1))


class ElasticityStudent(nn.Module):
    """Elasticity signal: EfficientNet-B0 + 14d handcrafted → elasticity score."""

    def __init__(self, handcrafted_dim: int = 14, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        cnn_dim = self.backbone.num_features

        self.hc_fc = nn.Sequential(
            nn.Linear(handcrafted_dim, 32),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.head = nn.Sequential(
            nn.Linear(cnn_dim + 32, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)
        hc_feat = self.hc_fc(handcrafted)
        return self.head(torch.cat([cnn_feat, hc_feat], dim=-1))


class MultiSignalStudent(nn.Module):
    """All 5 signals: EfficientNet-B0 + 58d handcrafted → 5 signal scores.

    Multi-task model that predicts all signals jointly. Cross-signal correlations
    (e.g., inflammation often correlates with lower structure) are learned implicitly.
    """

    def __init__(self, handcrafted_dim: int = 58, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        cnn_dim = self.backbone.num_features

        self.hc_fc = nn.Sequential(
            nn.Linear(handcrafted_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
        )

        self.shared = nn.Sequential(
            nn.Linear(cnn_dim + 128, 512),
            nn.GELU(),
            nn.Dropout(0.3),
        )

        # Per-signal heads (light specialization on top of shared representation)
        self.heads = nn.ModuleDict({
            'structure': nn.Linear(512, 1),
            'hydration': nn.Linear(512, 1),
            'inflammation': nn.Linear(512, 1),
            'sunDamage': nn.Linear(512, 1),
            'elasticity': nn.Linear(512, 1),
        })

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)
        hc_feat = self.hc_fc(handcrafted)
        shared = self.shared(torch.cat([cnn_feat, hc_feat], dim=-1))
        outputs = [self.heads[sig](shared) for sig in
                   ['structure', 'hydration', 'inflammation', 'sunDamage', 'elasticity']]
        return torch.cat(outputs, dim=-1)  # [B, 5]


# Print param counts
for name, cls, kwargs in [
    ('Structure', StructureStudent, {}),
    ('Hydration', HydrationStudent, {'handcrafted_dim': HYDRATION_DIM}),
    ('Elasticity', ElasticityStudent, {'handcrafted_dim': ELASTICITY_DIM}),
    ('MultiSignal', MultiSignalStudent, {'handcrafted_dim': MULTI_DIM}),
]:
    m = cls(pretrained=False, **kwargs)
    params = sum(p.numel() for p in m.parameters())
    print(f'{name:>15}: {params:>10,} params')

## 4. Training Engine

In [ ]:
def get_lr_scheduler(optimizer, num_epochs, warmup_epochs):
    """Cosine annealing with linear warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(num_epochs - warmup_epochs, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_epoch(model, loader, optimizer, loss_fn, signal_weights=None, uses_features=True):
    """Train one epoch. Returns average loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0

    for images, features, labels in loader:
        images = images.to(DEVICE)
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        if uses_features:
            preds = model(images, features)
        else:
            preds = model(images)

        # Per-signal weighted loss
        if signal_weights is not None and preds.shape[-1] == len(signal_weights):
            loss = sum(
                w * loss_fn(preds[:, i], labels[:, i])
                for i, w in enumerate(signal_weights)
            )
        else:
            loss = loss_fn(preds.squeeze(), labels.squeeze())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader, signal_names, uses_features=True):
    """Evaluate model. Returns per-signal MAE and Pearson r."""
    model.eval()
    all_preds = []
    all_labels = []

    for images, features, labels in loader:
        images = images.to(DEVICE)
        features = features.to(DEVICE)

        if uses_features:
            preds = model(images, features)
        else:
            preds = model(images)

        all_preds.append(preds.cpu())
        all_labels.append(labels)

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)

    metrics = {}
    for i, name in enumerate(signal_names):
        p = preds[:, i] if preds.dim() > 1 and preds.shape[1] > 1 else preds.squeeze()
        l = labels[:, i] if labels.dim() > 1 and labels.shape[1] > 1 else labels.squeeze()
        mae = torch.abs(p - l).mean().item()
        # Pearson correlation
        if p.std() > 0 and l.std() > 0:
            r = torch.corrcoef(torch.stack([p, l]))[0, 1].item()
        else:
            r = 0.0
        metrics[name] = {'mae': mae, 'pearson_r': r}

    return metrics


def train_model(model, train_loader, val_loader, signal_names,
                num_epochs=NUM_EPOCHS, lr=LR, uses_features=True,
                signal_weights=None, model_name='model'):
    """Full training loop with early stopping."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = get_lr_scheduler(optimizer, num_epochs, WARMUP_EPOCHS)
    loss_fn = nn.SmoothL1Loss()  # Huber loss — robust to teacher label noise

    best_mae = float('inf')
    best_state = None
    patience = 8
    patience_counter = 0
    history = {'train_loss': [], 'val_mae': [], 'val_r': []}

    print(f'\nTraining {model_name} for {num_epochs} epochs...')
    print(f'  Params: {sum(p.numel() for p in model.parameters()):,}')
    print(f'  Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}')

    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn,
                                 signal_weights, uses_features)
        val_metrics = evaluate(model, val_loader, signal_names, uses_features)
        scheduler.step()

        # Compute average MAE across all signals
        avg_mae = np.mean([m['mae'] for m in val_metrics.values()])
        avg_r = np.mean([m['pearson_r'] for m in val_metrics.values()])
        history['train_loss'].append(train_loss)
        history['val_mae'].append(avg_mae)
        history['val_r'].append(avg_r)

        lr_now = scheduler.get_last_lr()[0]
        per_signal = ' | '.join(f"{n}: {m['mae']:.1f}" for n, m in val_metrics.items())
        print(f'  [{epoch+1:>3}/{num_epochs}] loss={train_loss:.4f} | '
              f'MAE={avg_mae:.2f} | r={avg_r:.3f} | lr={lr_now:.6f}')
        if len(val_metrics) > 1:
            print(f'           {per_signal}')

        if avg_mae < best_mae:
            best_mae = avg_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f'           -> New best (MAE={best_mae:.2f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # Restore best weights
    if best_state:
        model.load_state_dict(best_state)

    # Save checkpoint
    ckpt_path = MODELS_DIR / f'{model_name}_best.pt'
    torch.save(best_state or model.state_dict(), ckpt_path)
    print(f'  Saved checkpoint: {ckpt_path}')

    return model, history, best_mae


print('Training engine ready.')

## 5. Train All Models

In [ ]:
# ── Load datasets ──────────────────────────────────────────────────
train_path = LABELS_DIR / 'train.json'
val_path = LABELS_DIR / 'val.json'

if not train_path.exists():
    print('No teacher labels found. Run notebook 10 first.')
else:
    print(f'Loading labels from {LABELS_DIR}...')
    with open(LABELS_DIR / 'dataset_info.json') as f:
        info = json.load(f)
    print(f'Dataset: {info["metadata"]["total_images"]} images')
    print(f'Splits: {info["metadata"]["splits"]}')
    print(f'Signal stats:')
    for sig, stats in info['signal_stats'].items():
        print(f'  {sig:>14}: mean={stats["mean"]:.1f}, std={stats["std"]:.1f}')

In [ ]:
# ── A. Train Structure Model ──────────────────────────────────────
# Input: green-channel CLAHE image (3-channel stacked)
# Output: [pore_count, texture_regularity, structure_score]
# Teacher only provides structure score — pore/texture are self-supervised auxiliaries

if train_path.exists():
    # Structure uses only the structure signal; pore/texture share the same label as proxy
    structure_train_ds = TeacherLabeledDataset(
        str(train_path), signals=['structure'],
        transform=train_transform, label_noise_std=LABEL_NOISE_STD,
        feature_mode='none',
    )
    structure_val_ds = TeacherLabeledDataset(
        str(val_path), signals=['structure'],
        transform=val_transform, feature_mode='none',
    )

    structure_train_loader = DataLoader(structure_train_ds, batch_size=BATCH_SIZE,
                                        shuffle=True, num_workers=NUM_WORKERS)
    structure_val_loader = DataLoader(structure_val_ds, batch_size=BATCH_SIZE,
                                      shuffle=False, num_workers=NUM_WORKERS)

    structure_model = StructureStudent(pretrained=True).to(DEVICE)

    # For structure, we train with the teacher's structure score for all 3 heads
    # (pore and texture are auxiliary, sharing the label as a multi-task regularizer)
    # Override the dataset to return 3 copies of the structure label
    class StructureDatasetWrapper(Dataset):
        def __init__(self, base_ds):
            self.base = base_ds
        def __len__(self):
            return len(self.base)
        def __getitem__(self, idx):
            img, feat, labels = self.base[idx]
            # Replicate structure score for pore and texture aux heads
            return img, feat, labels.expand(3)

    structure_train_wrapped = StructureDatasetWrapper(structure_train_ds)
    structure_val_wrapped = StructureDatasetWrapper(structure_val_ds)
    structure_train_loader = DataLoader(structure_train_wrapped, batch_size=BATCH_SIZE,
                                        shuffle=True, num_workers=NUM_WORKERS)
    structure_val_loader = DataLoader(structure_val_wrapped, batch_size=BATCH_SIZE,
                                      shuffle=False, num_workers=NUM_WORKERS)

    structure_model, structure_history, structure_best = train_model(
        structure_model, structure_train_loader, structure_val_loader,
        signal_names=['pore', 'texture', 'structure'],
        uses_features=False,
        signal_weights=[0.2, 0.2, 0.6],  # Structure head gets most weight
        model_name='structure',
    )

In [ ]:
# ── B. Train Hydration Model ──────────────────────────────────────

if train_path.exists():
    hydration_train_ds = TeacherLabeledDataset(
        str(train_path), signals=['hydration'],
        transform=train_transform, label_noise_std=LABEL_NOISE_STD,
        feature_mode='hydration',
    )
    hydration_val_ds = TeacherLabeledDataset(
        str(val_path), signals=['hydration'],
        transform=val_transform, feature_mode='hydration',
    )

    hydration_train_loader = DataLoader(hydration_train_ds, batch_size=BATCH_SIZE,
                                        shuffle=True, num_workers=NUM_WORKERS)
    hydration_val_loader = DataLoader(hydration_val_ds, batch_size=BATCH_SIZE,
                                      shuffle=False, num_workers=NUM_WORKERS)

    hydration_model = HydrationStudent(handcrafted_dim=HYDRATION_DIM, pretrained=True).to(DEVICE)

    hydration_model, hydration_history, hydration_best = train_model(
        hydration_model, hydration_train_loader, hydration_val_loader,
        signal_names=['hydration'],
        uses_features=True,
        model_name='hydration',
    )

In [ ]:
# ── C. Train Elasticity Model ─────────────────────────────────────

if train_path.exists():
    elasticity_train_ds = TeacherLabeledDataset(
        str(train_path), signals=['elasticity'],
        transform=train_transform, label_noise_std=LABEL_NOISE_STD,
        feature_mode='elasticity',
    )
    elasticity_val_ds = TeacherLabeledDataset(
        str(val_path), signals=['elasticity'],
        transform=val_transform, feature_mode='elasticity',
    )

    elasticity_train_loader = DataLoader(elasticity_train_ds, batch_size=BATCH_SIZE,
                                         shuffle=True, num_workers=NUM_WORKERS)
    elasticity_val_loader = DataLoader(elasticity_val_ds, batch_size=BATCH_SIZE,
                                       shuffle=False, num_workers=NUM_WORKERS)

    elasticity_model = ElasticityStudent(handcrafted_dim=ELASTICITY_DIM, pretrained=True).to(DEVICE)

    elasticity_model, elasticity_history, elasticity_best = train_model(
        elasticity_model, elasticity_train_loader, elasticity_val_loader,
        signal_names=['elasticity'],
        uses_features=True,
        model_name='elasticity',
    )

In [ ]:
# ── D. Train Multi-Signal Model ───────────────────────────────────
# This is a bonus model: predicts ALL 5 signals at once.
# Can replace the 3 individual models if it matches their accuracy.

if train_path.exists():
    multi_train_ds = TeacherLabeledDataset(
        str(train_path), signals=SIGNALS,
        transform=train_transform, label_noise_std=LABEL_NOISE_STD,
        feature_mode='all',
    )
    multi_val_ds = TeacherLabeledDataset(
        str(val_path), signals=SIGNALS,
        transform=val_transform, feature_mode='all',
    )

    multi_train_loader = DataLoader(multi_train_ds, batch_size=BATCH_SIZE,
                                     shuffle=True, num_workers=NUM_WORKERS)
    multi_val_loader = DataLoader(multi_val_ds, batch_size=BATCH_SIZE,
                                   shuffle=False, num_workers=NUM_WORKERS)

    multi_model = MultiSignalStudent(handcrafted_dim=MULTI_DIM, pretrained=True).to(DEVICE)

    # Equal weights across signals
    multi_model, multi_history, multi_best = train_model(
        multi_model, multi_train_loader, multi_val_loader,
        signal_names=SIGNALS,
        uses_features=True,
        signal_weights=[0.2, 0.2, 0.2, 0.2, 0.2],
        model_name='multi_signal',
    )

## 6. ONNX Export

Export models in the exact format expected by `backend/signal-models.js`.

In [ ]:
import onnxruntime as ort


def export_structure_onnx(model, output_path):
    """Export structure model. Backend expects input='image', output='scores' [B,3]."""
    model.eval().cpu()
    dummy = torch.randn(1, 3, 224, 224)
    torch.onnx.export(
        model, dummy, output_path,
        input_names=['image'],
        output_names=['scores'],
        dynamic_axes={'image': {0: 'batch'}, 'scores': {0: 'batch'}},
        opset_version=17,
    )
    # Verify
    sess = ort.InferenceSession(output_path)
    out = sess.run(None, {'image': dummy.numpy()})
    print(f'Structure ONNX: shape={out[0].shape}, sample={out[0][0]}')
    return output_path


def export_signal_onnx(model, output_path, signal_name, handcrafted_dim):
    """Export hydration/elasticity model. Backend expects image + handcrafted_features."""
    model.eval().cpu()
    dummy_img = torch.randn(1, 3, 224, 224)
    dummy_hc = torch.randn(1, handcrafted_dim)
    torch.onnx.export(
        model, (dummy_img, dummy_hc), output_path,
        input_names=['image', 'handcrafted_features'],
        output_names=[f'{signal_name}_score'],
        dynamic_axes={
            'image': {0: 'batch'},
            'handcrafted_features': {0: 'batch'},
            f'{signal_name}_score': {0: 'batch'},
        },
        opset_version=17,
    )
    # Verify
    sess = ort.InferenceSession(output_path)
    out = sess.run(None, {'image': dummy_img.numpy(), 'handcrafted_features': dummy_hc.numpy()})
    print(f'{signal_name} ONNX: shape={out[0].shape}, sample={out[0][0]}')
    return output_path


def export_multi_onnx(model, output_path, handcrafted_dim):
    """Export multi-signal model."""
    model.eval().cpu()
    dummy_img = torch.randn(1, 3, 224, 224)
    dummy_hc = torch.randn(1, handcrafted_dim)
    torch.onnx.export(
        model, (dummy_img, dummy_hc), output_path,
        input_names=['image', 'handcrafted_features'],
        output_names=['signal_scores'],
        dynamic_axes={
            'image': {0: 'batch'},
            'handcrafted_features': {0: 'batch'},
            'signal_scores': {0: 'batch'},
        },
        opset_version=17,
    )
    sess = ort.InferenceSession(output_path)
    out = sess.run(None, {'image': dummy_img.numpy(), 'handcrafted_features': dummy_hc.numpy()})
    print(f'MultiSignal ONNX: shape={out[0].shape}, sample={out[0][0]}')
    return output_path


# Export all models
if train_path.exists():
    export_dir = MODELS_DIR / 'onnx'
    export_dir.mkdir(exist_ok=True)

    export_structure_onnx(structure_model, str(export_dir / 'structure.onnx'))
    export_signal_onnx(hydration_model, str(export_dir / 'hydration.onnx'),
                       'hydration', HYDRATION_DIM)
    export_signal_onnx(elasticity_model, str(export_dir / 'elasticity.onnx'),
                       'elasticity', ELASTICITY_DIM)
    export_multi_onnx(multi_model, str(export_dir / 'multi_signal.onnx'), MULTI_DIM)

    # Print file sizes
    for f in export_dir.glob('*.onnx'):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  {f.name}: {size_mb:.1f} MB')
else:
    print('No models to export. Train first.')

## 7. Training Visualization

In [ ]:
SIGNAL_COLORS_LIST = ['#7DE7E1', '#4DA6FF', '#FF7A78', '#F2B56A', '#B68AFF']


def plot_training_curves(histories: dict[str, dict]):
    """Plot loss and MAE curves for all models."""
    n = len(histories)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)

    for i, (name, hist) in enumerate(histories.items()):
        color = SIGNAL_COLORS_LIST[i % len(SIGNAL_COLORS_LIST)]

        # Loss
        axes[0, i].plot(hist['train_loss'], color=color, linewidth=2)
        axes[0, i].set_title(f'{name} - Loss')
        axes[0, i].set_xlabel('Epoch')
        axes[0, i].set_ylabel('Smooth L1 Loss')
        axes[0, i].grid(True, alpha=0.3)

        # Validation MAE
        axes[1, i].plot(hist['val_mae'], color=color, linewidth=2, label='MAE')
        axes[1, i].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Target (10)')
        axes[1, i].set_title(f'{name} - Val MAE')
        axes[1, i].set_xlabel('Epoch')
        axes[1, i].set_ylabel('MAE')
        axes[1, i].legend()
        axes[1, i].grid(True, alpha=0.3)

    plt.suptitle('Distillation Training Curves', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(MODELS_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()


if train_path.exists():
    plot_training_curves({
        'Structure': structure_history,
        'Hydration': hydration_history,
        'Elasticity': elasticity_history,
        'MultiSignal': multi_history,
    })

## Summary

After running this notebook you should have:
- `models/distilled/{structure,hydration,elasticity,multi_signal}_best.pt` — PyTorch checkpoints
- `models/distilled/onnx/{structure,hydration,elasticity,multi_signal}.onnx` — ONNX for backend
- `models/distilled/training_curves.png` — visualization

Copy the ONNX files to `RadianceIQ/backend/models/` to deploy:
```bash
cp models/distilled/onnx/structure.onnx ../RadianceIQ/backend/models/
cp models/distilled/onnx/hydration.onnx ../RadianceIQ/backend/models/
cp models/distilled/onnx/elasticity.onnx ../RadianceIQ/backend/models/
```

**Next:** Use `12_distillation_eval.ipynb` to evaluate student vs teacher agreement.